In [22]:
import pickle
import numpy as np
import pandas as pd

In [23]:
# Loop through all three batches
for batch_num in [1, 2, 3]:
    print(f"BATCH {batch_num}")
    
    # Load the batch
    bat_dict = pickle.load(open(f'batch{batch_num}.pkl', 'rb'))
    
    # Quick overview
    print(f"Total batteries: {len(bat_dict)}")
    print(f"\nBattery names: {list(bat_dict.keys())[:]}")
    
    # Look at one battery
    first_battery_name = list(bat_dict.keys())[0]
    sample_battery = bat_dict[first_battery_name]
    print(f"\nStructure of {first_battery_name}:")
    print(f"Keys: {sample_battery.keys()}\n")

BATCH 1
Total batteries: 46

Battery names: ['b1c0', 'b1c1', 'b1c2', 'b1c3', 'b1c4', 'b1c5', 'b1c6', 'b1c7', 'b1c8', 'b1c9', 'b1c10', 'b1c11', 'b1c12', 'b1c13', 'b1c14', 'b1c15', 'b1c16', 'b1c17', 'b1c18', 'b1c19', 'b1c20', 'b1c21', 'b1c22', 'b1c23', 'b1c24', 'b1c25', 'b1c26', 'b1c27', 'b1c28', 'b1c29', 'b1c30', 'b1c31', 'b1c32', 'b1c33', 'b1c34', 'b1c35', 'b1c36', 'b1c37', 'b1c38', 'b1c39', 'b1c40', 'b1c41', 'b1c42', 'b1c43', 'b1c44', 'b1c45']

Structure of b1c0:
Keys: dict_keys(['cycle_life', 'charge_policy', 'summary', 'cycles'])

BATCH 2
Total batteries: 48

Battery names: ['b2c0', 'b2c1', 'b2c2', 'b2c3', 'b2c4', 'b2c5', 'b2c6', 'b2c7', 'b2c8', 'b2c9', 'b2c10', 'b2c11', 'b2c12', 'b2c13', 'b2c14', 'b2c15', 'b2c16', 'b2c17', 'b2c18', 'b2c19', 'b2c20', 'b2c21', 'b2c22', 'b2c23', 'b2c24', 'b2c25', 'b2c26', 'b2c27', 'b2c28', 'b2c29', 'b2c30', 'b2c31', 'b2c32', 'b2c33', 'b2c34', 'b2c35', 'b2c36', 'b2c37', 'b2c38', 'b2c39', 'b2c40', 'b2c41', 'b2c42', 'b2c43', 'b2c44', 'b2c45', 'b2c46', 'b

In [25]:
# Load all batches
batch1 = pickle.load(open('batch1.pkl', 'rb'))
batch2 = pickle.load(open('batch2.pkl', 'rb'))
batch3 = pickle.load(open('batch3.pkl', 'rb'))

In [26]:
# Merge all batches
bat_dict = {**batch1, **batch2, **batch3}

In [27]:
# Count total before filtering
total_batteries = len(bat_dict)

In [28]:
# Extract cycle life for ALL batteries
cycle_lives = []
battery_names = []
batch_nums = []
bad_batteries = []

for bat_name, bat_data in bat_dict.items():
    cycle_life = bat_data['cycle_life']
    
    # Flatten the value regardless of format
    if isinstance(cycle_life, (list, np.ndarray)):
        cycle_life = float(np.array(cycle_life).flatten()[0])
    else:
        cycle_life = float(cycle_life)
    
    # Check if value is NaN or invalid
    if np.isnan(cycle_life) or cycle_life <= 0:
        bad_batteries.append(bat_name)
        continue
    
    cycle_lives.append(cycle_life)
    battery_names.append(bat_name)
    
    # Extract batch number from battery name
    batch_num = int(bat_name[1])
    batch_nums.append(batch_num)

In [29]:
# Create DataFrame with only valid batteries
merged_df = pd.DataFrame({
    'Battery': battery_names,
    'Batch': batch_nums,
    'Cycle_Life': cycle_lives
})

In [30]:
# Report filtered batteries
print(f"Filtered out {len(bad_batteries)} batteries with NaN/invalid values:")
print(f"   {bad_batteries}\n")

Filtered out 2 batteries with NaN/invalid values:
   ['b3c23', 'b3c32']



# Extracting a stratified sub-dataset
In this section, we will extract a smaller dataset of sample experiments across the three batches.
We chose a "stratified" strategy, in which we will divide the batteries into 3 groups by cycle life (bottom 33%, middle 33%, top 33%).
We will use the merged dataset obtained from above.

In [ ]:
# Stratified sampling
# Define cycle life ranges
short_threshold = merged_df['Cycle_Life'].quantile(0.33)
long_threshold = merged_df['Cycle_Life'].quantile(0.67)

print(f"\nStratification thresholds:")
print(f"Short: < {short_threshold:.0f} cycles")
print(f"Medium: {short_threshold:.0f} - {long_threshold:.0f} cycles")
print(f"Long: > {long_threshold:.0f} cycles")

# Create stratified groups
short = merged_df[merged_df['Cycle_Life'] < short_threshold]
medium = merged_df[(merged_df['Cycle_Life'] >= short_threshold) & 
                   (merged_df['Cycle_Life'] < long_threshold)]
long = merged_df[merged_df['Cycle_Life'] >= long_threshold]

print(f"\nGroup sizes:")
print(f"Short: {len(short)} batteries")
print(f"Medium: {len(medium)} batteries")
print(f"Long: {len(long)} batteries")

# Sample from each group (sample(n) picks n random batteries)
n_per_group = 5 

short_sample = short.sample(n=min(n_per_group, len(short)), random_state=42) 
medium_sample = medium.sample(n=min(n_per_group, len(medium)), random_state=42)
long_sample = long.sample(n=min(n_per_group, len(long)), random_state=42)

# Combine into subset
subset_df = pd.concat([short_sample, medium_sample, long_sample]).sort_values('Cycle_Life')

print(f"Selected {len(subset_df)} batteries for subset")

# === EXPORT ALL DATA FROM SELECTED BATTERIES ===
all_data = []

for bat_name in subset_df['Battery']:
    bat_data = bat_dict[bat_name]
    
    # Get battery metadata
    cycle_life = bat_data['cycle_life']
    if isinstance(cycle_life, (list, np.ndarray)):
        cycle_life = float(np.array(cycle_life).flatten()[0])
    
    policy = bat_data['charge_policy']
    
    # Get summary data
    summary = bat_data['summary']
    
    # Create a row for each cycle
    for i in range(len(summary['cycle'])):
        row = {
            'Battery': bat_name,
            'Cycle_Life': cycle_life,
            'Policy': policy,
            'Cycle': int(summary['cycle'][i]),
            'IR': summary['IR'][i],
            'QC': summary['QC'][i],
            'QD': summary['QD'][i],
            'Tavg': summary['Tavg'][i],
            'Tmin': summary['Tmin'][i],
            'Tmax': summary['Tmax'][i],
            'chargetime': summary['chargetime'][i]
        }
        all_data.append(row)

# Create DataFrame and save
data_df = pd.DataFrame(all_data)

# Save to CSV
data_df.to_csv('battery_subset_data.csv', index=False)

print(f"\nSaved to 'battery_subset_data.csv'")
print(f"Total rows: {len(data_df)} (one per cycle per battery)")
print(f"\nFirst few rows:")
print(data_df.head(10))

print(f"\nColumns: {list(data_df.columns)}")
print(f"\nFile size: ~{len(data_df) * 0.0005:.1f}MB (calculated as 0.5KB per row)")

# Also save the battery list separately
subset_df.to_csv('battery_subset_list.csv', index=False)
print(f"\nAlso saved 'battery_subset_list.csv' (battery names and cycle life)")